# A02 — Instrumentos lineales: depósitos, FRAs, OIS y futuros

**Objetivos:** valorar y liquidar un FRA; capitalizar €STR; ajuste de convexidad de futuros; caso de la caja de 30 M€.

**Fuentes:** Hull cap. 4, 6 y 9; Veronesi cap. 5–6; Schofield & Bowler cap. 1–2. Datos sintéticos.

Las curvas se construyen en A03; aquí se usan como caja negra (`build_curves`).

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent          # notebooks/ -> repository root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from datetime import date
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

from derivlab.market import sample_market
from derivlab.curves import build_curves, forward_rate, discount, tenor_end
from derivlab.dates import spot_date, year_fraction, add_business_days, is_business_day
from derivlab.swaps import fra_settlement, fra_value

m = sample_market()
estr, eur = build_curves(m)
ref, spot = m["ref_date"], spot_date(m["ref_date"])
print("Reference date:", ref, " spot:", spot)

Reference date: 2026-10-26  spot: 2026-10-28


## 1. FRA 3x9 sobre Euribor 6M

In [2]:
fra_start = tenor_end(spot, "3M")
fra_end = tenor_end(fra_start, "6M")
tau = year_fraction(fra_start, fra_end, "ACT/360")
K = forward_rate(eur, fra_start, fra_end)
print(f"FRA 3x9: {fra_start} -> {fra_end}, tau = {tau:.6f}, fair rate = {K:.4%}")
print(f"Value at fair rate: {fra_value(estr, eur, fra_start, fra_end, 30e6, K):.6f}")

FRA 3x9: 2027-01-28 -> 2027-07-28, tau = 0.502778, fair rate = 2.0354%
Value at fair rate: 0.000000


## 2. Caso: fijar hoy la rentabilidad de 30 M€ que llegan dentro de 3 meses

La empresa **vende** el FRA (recibe fijo K, paga Euribor). En la fecha de fixing deposita la caja al Euribor
del momento y reinvierte la liquidación del FRA. El resultado debe ser exactamente K, porque
$(N + S)(1 + L\tau) = N(1 + K\tau)$ cuando $S = -N(L-K)\tau/(1+L\tau)$.

In [3]:
N = 30_000_000
rows = []
for L in [0.0150, K, 0.0260]:
    S = fra_settlement(N, L, K, tau, buyer=False)
    total = (N + S) * (1 + L * tau)
    rows.append({"euribor_fixing": L, "fra_settlement": S, "cash_at_end": total,
                 "effective_rate": (total / N - 1) / tau})
case = pd.DataFrame(rows)
print(case.to_string(formatters={"euribor_fixing": "{:.4%}".format, "effective_rate": "{:.4%}".format,
                                 "fra_settlement": "{:,.0f}".format, "cash_at_end": "{:,.0f}".format}))
margin_bp = 3
cost = -fra_value(estr, eur, fra_start, fra_end, N, K - margin_bp * 1e-4, buyer=False)
print(f"CASE FRA rate = {K:.4%}; cost today of a {margin_bp} bp worse strike = {cost:,.0f} EUR")

  euribor_fixing fra_settlement cash_at_end effective_rate
0        1.5000%         80,148  30,307,002        2.0354%
1        2.0354%             -0  30,307,002        2.0354%
2        2.6000%        -84,066  30,307,002        2.0354%
CASE FRA rate = 2.0354%; cost today of a 3 bp worse strike = 4,457 EUR


## 3. €STR capitalizado

El tipo de un OIS se calcula capitalizando el €STR diario; cada fixing pesa por los días naturales hasta el
siguiente día hábil: $R = \left[\prod_i (1 + r_i d_i/360) - 1\right]\cdot 360/D$.
Supuesto ilustrativo: el BCE baja 25 pb con efecto el 17-12-2026.

In [4]:
def compounded_rate(start, end, rate_fn):
    d, growth, days_total = start, 1.0, (end - start).days
    while d < end:
        nxt = add_business_days(d, 1)
        nxt = min(nxt, end)
        growth *= 1 + rate_fn(d) * (nxt - d).days / 360
        d = nxt
    return (growth - 1) * 360 / days_total

def weighted_average(start, end, rate_fn):
    d, acc = start, 0.0
    while d < end:
        nxt = min(add_business_days(d, 1), end)
        acc += rate_fn(d) * (nxt - d).days
        d = nxt
    return acc / (end - start).days

cut = date(2026, 12, 17)
path = lambda d: 0.01874 if d < cut else 0.01624
p_start, p_end = spot, tenor_end(spot, "3M")
r_comp = compounded_rate(p_start, p_end, path)
r_avg = weighted_average(p_start, p_end, path)
print(f"CASE compounded €STR 3M = {r_comp:.5%}, weighted average = {r_avg:.5%}, "
      f"difference = {(r_comp - r_avg) * 1e4:.2f} bp")

CASE compounded €STR 3M = 1.76375%, weighted average = 1.75987%, difference = 0.39 bp


## 4. Futuros y ajuste de convexidad

Con un modelo Ho-Lee (Hull cap. 6): $F_{fwd} = F_{fut} - \tfrac12\sigma^2 t_1 t_2$, con σ la volatilidad
normal anual del tipo corto. El ajuste crece con el cuadrado del plazo.

In [5]:
sigma = 0.009
conv = pd.DataFrame({"t1": np.arange(1, 9)})
conv["t2"] = conv.t1 + 0.25
conv["adjustment_bp"] = 0.5 * sigma ** 2 * conv.t1 * conv.t2 * 1e4
print(conv.round(2).to_string(index=False))
print(f"CASE convexity adjustment at 4y: {conv.adjustment_bp.iloc[3]:.1f} bp")

 t1   t2  adjustment_bp
  1 1.25           0.51
  2 2.25           1.82
  3 3.25           3.95
  4 4.25           6.88
  5 5.25          10.63
  6 6.25          15.19
  7 7.25          20.55
  8 8.25          26.73
CASE convexity adjustment at 4y: 6.9 bp


## 5. Comprobaciones

In [6]:
assert np.allclose(case.effective_rate, K, atol=1e-12)          # the FRA hedge is exact
flat = lambda d: 0.02
growth_check = compounded_rate(p_start, p_end, flat)
assert growth_check > 0.02                                       # compounding beats simple
assert r_comp > r_avg
assert np.all(np.diff(conv.adjustment_bp) > 0)
print("All checks passed")

All checks passed


## Ejercicios

1. **Básico.** Un comprador de FRA sobre 10 M€ con strike 2,00 % ve un fixing de 2,40 % para τ = 0,5. ¿Cuánto cobra en la fecha de inicio?
2. **Intermedio.** Calcula el €STR capitalizado entre el 23-12-2026 y el 05-01-2027 con un €STR constante del 1,874 %. ¿Cuántos días pesa el fixing del 24-12?
3. **Avanzado.** Ocho futuros trimestrales cotizan a 98,00; 97,90; 97,82; 97,75; 97,70; 97,66; 97,63; 97,60 (vencimientos 0,25 a 2 años). Convierte cada precio a tipo forward con σ = 0,9 %.

## Soluciones

In [7]:
print(f"{fra_settlement(10e6, 0.024, 0.020, 0.5, buyer=True):,.2f} EUR")

19,762.85 EUR


In [8]:
a, b = date(2026, 12, 23), date(2027, 1, 5)
print(f"compounded: {compounded_rate(a, b, lambda d: 0.01874):.6%}")
print("24-12 weighs", (add_business_days(date(2026, 12, 24), 1) - date(2026, 12, 24)).days, "days")

compounded: 1.874495%
24-12 weighs 4 days


In [9]:
prices = np.array([98.00, 97.90, 97.82, 97.75, 97.70, 97.66, 97.63, 97.60])
t1 = np.arange(0.25, 2.01, 0.25)
fut_rate = (100 - prices) / 100
fwd = fut_rate - 0.5 * 0.009 ** 2 * t1 * (t1 + 0.25)
pd.DataFrame({"t1": t1, "futures_rate": fut_rate, "forward_rate": fwd,
              "adj_bp": (fut_rate - fwd) * 1e4}).round(5)

,t1,futures_rate,forward_rate,adj_bp
0,0.25,0.0200,0.01999,0.05062
1,0.50,0.0210,0.02098,0.15187
2,0.75,0.0218,0.02177,0.30375
3,1.00,0.0225,0.02245,0.50625
4,1.25,0.0230,0.02292,0.75938
5,1.50,0.0234,0.02329,1.06313
6,1.75,0.0237,0.02356,1.41750
7,2.00,0.0240,0.02382,1.82250
